# Price Prediction Models

## Load data

In [5]:
import pandas as pd
import numpy as np

resale = pd.read_csv("Data/processed.csv")
resale.head()

,month,town,flat_type,floor_area_sqm,remaining_lease,resale_price,latitude,longitude,storey_type_lower,storey_type_middle,...,rpi,adjusted_resale,mrt_count,school_count,mall_count,bus_stop_count,hawker_count,wet_market_count,nearest_mrt_dist,nearest_bus_stop_dist
0,0,ANG MO KIO,2 ROOM,44.0,736,232000.0,1.362005,103.853880,False,False,...,133.9,173263.629574,0,3,0,14,1,0,1016,91
1,0,ANG MO KIO,3 ROOM,67.0,727,250000.0,1.370966,103.838202,True,False,...,133.9,186706.497386,1,2,0,11,1,0,202,164
2,0,ANG MO KIO,3 ROOM,67.0,749,262000.0,1.380709,103.835368,True,False,...,133.9,195668.409261,1,0,0,12,0,0,460,136
3,0,ANG MO KIO,3 ROOM,68.0,745,265000.0,1.366201,103.857201,False,True,...,133.9,197908.887229,0,1,0,10,2,0,828,68
4,0,ANG MO KIO,3 ROOM,67.0,749,265000.0,1.381041,103.835132,True,False,...,133.9,197908.887229,1,0,0,11,0,0,433,146


In [6]:
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Separate features and target
resale = resale.dropna(subset=["nearest_mrt_dist", "nearest_bus_stop_dist"])
X = resale.drop(columns=['floor_area_sqm', 'adjusted_resale', 'resale_price', 'latitude', 'longitude', 'month', 'rpi'])
y = resale["adjusted_resale"]

# Convert categorical variables
X = pd.get_dummies(X, drop_first=True)

# Create 5 bins based on price distribution  (low, medium, high for 3 bins)
y_binned = pd.qcut(y, q=5, labels=False)

# First split: Train (60%) and Temp (40%)
X_train, X_temp, y_train, y_temp, yb_train, yb_temp = train_test_split(
    X, y, y_binned,
    test_size=0.4,
    stratify=y_binned,
    random_state=42
)

# Second split: Validation (20%) and Test (20%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    stratify=yb_temp,
    random_state=42
)

# Check sizes
print("Training set:", X_train.shape)
print("Validation set:", X_val.shape)
print("Test set:", X_test.shape)

Training set: (134725, 43)
Validation set: (44908, 43)
Test set: (44909, 43)


## Dummy Regressor

In [7]:
# Initialize dummy model (predicts mean of y_train)
dummy_model = DummyRegressor(strategy="mean")

# Train
dummy_model.fit(X_train, y_train)

# Predict on test set
dummy_pred = dummy_model.predict(X_test)

# Evaluate
print("Dummy Baseline Performance:")
print("MAE:", mean_absolute_error(y_test, dummy_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, dummy_pred)))
print("R2:", r2_score(y_test, dummy_pred))

Dummy Baseline Performance:
MAE: 83094.00326267991
RMSE: 108116.49177632012
R2: -2.7113559619706962e-08


## Linear Regression

In [8]:
# Initialize model
lr_model = LinearRegression()

# Train
lr_model.fit(X_train, y_train)

# Predict
lr_pred = lr_model.predict(X_test)

# Evaluate
print("Linear Regression Performance:")
print("MAE:", mean_absolute_error(y_test, lr_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, lr_pred)))
print("R2:", r2_score(y_test, lr_pred))

# 5-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
lr_r2, lr_mae, lr_rmse = [], [], []

for train_idx, val_idx in kf.split(X_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    lr_cv = LinearRegression()
    lr_cv.fit(X_tr, y_tr)
    pred = lr_cv.predict(X_val)

    lr_r2.append(r2_score(y_val, pred))
    lr_mae.append(mean_absolute_error(y_val, pred))
    lr_rmse.append(np.sqrt(mean_squared_error(y_val, pred)))

print("\nLinear Regression Cross Validation Results")
print("Mean R2:", np.mean(lr_r2))
print("Std R2:", np.std(lr_r2))
print("Mean MAE:", np.mean(lr_mae))
print("Mean RMSE:", np.mean(lr_rmse))

Linear Regression Performance:
MAE: 34453.13070541199
RMSE: 45685.20993000975
R2: 0.8214469095411567

Linear Regression Cross Validation Results
Mean R2: 0.8192416581152486
Std R2: 0.003125464799931236
Mean MAE: 34505.47406150297
Mean RMSE: 45803.70452114835


## Random Forest

In [ ]:
# Init the RF model
rf_model = RandomForestRegressor(random_state=42)

# Train
rf_model.fit(X_train, y_train)

# Predict
rf_pred = rf_model.predict(X_test)

# Evaluate
print("Random Forest Performance:"
      f"MAE: ${mean_absolute_error(y_test, rf_pred):.0f} \n" 
      f"RMSE: ${np.sqrt(mean_squared_error(y_test, rf_pred)):.0f} \n"
      f"R2: {r2_score(y_test, rf_pred):.6f}")

# 5-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
rf_r2, rf_mae, rf_rmse = [], [], []

for train_idx, val_idx in kf.split(X_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    rf_cv = RandomForestRegressor(random_state=42, n_jobs=-1)
    rf_cv.fit(X_tr, y_tr)
    pred = rf_cv.predict(X_val)

    rf_r2.append(r2_score(y_val, pred))
    rf_mae.append(mean_absolute_error(y_val, pred))
    rf_rmse.append(np.sqrt(mean_squared_error(y_val, pred)))

rf_base_r2 = np.mean(rf_r2)
rf_base_r2_std = np.std(rf_r2)
rf_base_mae = np.mean(rf_mae)
rf_base_mae_std = np.std(rf_mae)
rf_base_rmse = np.mean(rf_rmse)
rf_base_rmse_std = np.std(rf_rmse)

print("\nRandom Forest Cross Validation Results \n"
        f"Mean R2: {rf_base_r2:.4f} ± {rf_base_r2_std:.6f} \n"
        f"Mean MAE: ${rf_base_mae:.0f} ± ${rf_base_mae_std:.2f}\n"
        f"Mean RMSE: ${rf_base_rmse:.0f} ± ${rf_base_rmse_std:.2f}")

Random Forest Performance:MAE: $15366.453389893983 
RMSE: $22263.35690913777 
R2: 0.9575969184622676

Random Forest Cross Validation Results 
Mean R2: 0.9553 ± 0.000992 
Mean MAE: $15701.873763 ± $60.474755
Mean RMSE: $22784.880945 ± $229.266178


# RPI

## RPI Calculation

In [52]:
rpi = pd.read_csv('Data/2025-RPI.csv')
avg_rpi = rpi['rpi'].diff().mean()
avg_rpi = round(avg_rpi, 3)

BASE_RPI = 133.9  # Base RPI as of 2017-01
print(avg_rpi)

1.521


## Metrics with RPI added in

### Preparing the data

In [64]:
y_actual = resale["resale_price"]
X = resale.drop(columns=['floor_area_sqm', 'adjusted_resale', 'resale_price', 'latitude', 'longitude', 'rpi'])

# Convert categorical variables
X = pd.get_dummies(X, drop_first=True)

# Create 5 bins based on price distribution  (low, medium, high for 3 bins)
y_binned = pd.qcut(y, q=5, labels=False)

# First split: Train (60%) and Temp (40%)
X_act_train, X_act_temp, y_act_train, y_act_temp, yb_train, yb_temp = train_test_split(
    X, y_actual, y_binned,
    test_size=0.4,
    stratify=y_binned,
    random_state=42
)

# Second split: Validation (20%) and Test (20%)
X_act_val, X_act_test, y_act_val, y_act_test = train_test_split(
    X_act_temp, y_act_temp,
    test_size=0.5,
    stratify=yb_temp,
    random_state=42
)

month_x = X_act_test[['month']].copy()
X_act_test = X_act_test.drop(columns=['month'])

### Linear Regression

In [54]:
# Get the predicated value using the model already trained above
lr_test = X_act_test.copy()
lr_pred = lr_model.predict(lr_test)

lr_rpi_test = pd.DataFrame()
lr_rpi_test.index = lr_test.index.copy()
lr_rpi_test["Pred"] = lr_pred

months = month_x.copy()
lr_rpi_test["month"] = months
lr_rpi_test["Pred_Actual"] = lr_rpi_test['Pred'] * ((lr_rpi_test["month"] // 3 * avg_rpi + BASE_RPI) / 100)

# Evaluate
print("Linear Regression Performance:")
print("MAE:", mean_absolute_error(y_act_test, lr_rpi_test["Pred_Actual"]))
print("RMSE:", np.sqrt(mean_squared_error(y_act_test, lr_rpi_test["Pred_Actual"])))
print("R2:", r2_score(y_act_test, lr_rpi_test["Pred_Actual"]))

Linear Regression Performance:
MAE: 66403.77001057025
RMSE: 84495.49536623401
R2: 0.7979370329881028


#### Cross validation

In [ ]:
# 5-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
lr_r2, lr_mae, lr_rmse = [], [], []

for train_idx, val_idx in kf.split(X_act_train):
    X_tr, X_val = X_act_train.iloc[train_idx], X_act_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_act_train.iloc[val_idx]

    lr_cv = LinearRegression()
    months = X_val['month'].copy()
    X_tr = X_tr.drop(columns=['month'])
    X_val = X_val.drop(columns=['month'])
    lr_cv.fit(X_tr, y_tr)
    pred = lr_cv.predict(X_val)

    lr_rpi_test = pd.DataFrame()
    lr_rpi_test.index = X_val.index.copy()
    lr_rpi_test["Pred"] = pred
    lr_rpi_test["month"] = months
    pred_actual = lr_rpi_test['Pred'] * ((lr_rpi_test["month"] // 3 * avg_rpi + BASE_RPI) / 100)

    lr_r2.append(r2_score(y_val, pred_actual))
    lr_mae.append(mean_absolute_error(y_val, pred_actual))
    lr_rmse.append(np.sqrt(mean_squared_error(y_val, pred_actual)))

print("Linear Regression Cross Validation Results")
print("Mean R2:", np.mean(lr_r2))
print("Std R2:", np.std(lr_r2))
print("Mean MAE:", np.mean(lr_mae))
print("Mean RMSE:", np.mean(lr_rmse))


Linear Regression Cross Validation Results
Mean R2: 0.7959971300811952
Std R2: 0.0031572880297890574
Mean MAE: 66562.52409191985
Mean RMSE: 84584.39980761516


### Random Forest

In [65]:
rf_pred = rf_model.predict(X_act_test)

months = month_x.copy()

rf_rpi_test = pd.DataFrame()
rf_rpi_test.index = X_act_test.index.copy()
rf_rpi_test["Pred"] = rf_pred
rf_rpi_test["month"] = months
rf_rpi_test["Pred_Actual"] = rf_rpi_test['Pred'] * ((rf_rpi_test["month"] // 3 * avg_rpi + BASE_RPI) / 100)

# Evaluate
print("Random Forest Performance: \n"
        f"MAE: ${mean_absolute_error(y_act_test, rf_rpi_test["Pred_Actual"]):,.0f} \n"
        f"RMSE: ${np.sqrt(mean_squared_error(y_act_test, rf_rpi_test["Pred_Actual"])):,.0f} \n"
        f"R2: {r2_score(y_act_test, rf_rpi_test["Pred_Actual"]):.4f}")

Random Forest Performance: 
MAE: $43,461 
RMSE: $55,895 
R2: 0.9116


In [66]:
overshot = (y_act_test < rf_rpi_test['Pred_Actual']).sum()
undershot = (y_act_test > rf_rpi_test['Pred_Actual']).sum()

print(f"Overshot: {overshot} \nUndershot: {undershot}")

Overshot: 25930 
Undershot: 18979


#### Cross validation

In [ ]:
# 5-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
rf_r2, rf_mae, rf_rmse = [], [], []

for train_idx, val_idx in kf.split(X_act_train):
    X_tr, X_val = X_act_train.iloc[train_idx], X_act_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_act_train.iloc[val_idx]

    rf_cv = RandomForestRegressor(random_state=42, n_jobs=-1)
    month_x = X_val['month'].copy()
    X_tr = X_tr.drop(columns=['month'])
    X_val = X_val.drop(columns=['month'])
    rf_cv.fit(X_tr, y_tr)
    pred = rf_cv.predict(X_val)

    rf_rpi_test = pd.DataFrame()
    rf_rpi_test.index = X_val.index.copy()
    rf_rpi_test["Pred"] = pred
    rf_rpi_test["month"] = month_x
    pred_actual = rf_rpi_test['Pred'] * ((rf_rpi_test["month"] // 3 * avg_rpi + BASE_RPI) / 100)

    rf_r2.append(r2_score(y_val, pred_actual))
    rf_mae.append(mean_absolute_error(y_val, pred_actual))
    rf_rmse.append(np.sqrt(mean_squared_error(y_val, pred_actual)))

print("Random Forest Cross Validation Results \n"
        f"Mean R2: {np.mean(rf_r2):.4f} ± {np.std(rf_r2):.6f} \n"
        f"Mean MAE: ${np.mean(rf_mae):,.0f} ± ${np.std(rf_mae):,.0f} \n"
        f"Mean RMSE: ${np.mean(rf_rmse):,.0f} ± ${np.std(rf_rmse):,.0f}")

Random Forest Cross Validation Results 
Mean R2: 0.9097 ± 0.001468 
Mean MAE: $43,740 ± 186 
Mean RMSE: $56,261 ± 336


### HyperTuning (using the RPI metrics)

In [58]:
from sklearn.utils import resample
from sklearn.model_selection import RandomizedSearchCV

def run_cv_rpi(X_tr_full, y_tr_full, model_params):
    r2s, maes = [], []
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    for train_idx, val_idx in kf.split(X_tr_full):
        X_tr = X_tr_full.iloc[train_idx]
        X_val = X_tr_full.iloc[val_idx]
        y_tr = y_tr_full.iloc[train_idx]

        months_val = X_val['month'].copy()
        X_tr = X_tr.drop(columns=['month'])
        X_val = X_val.drop(columns=['month'])

        m = RandomForestRegressor(**model_params)
        m.fit(X_tr, y_tr)
        pred = m.predict(X_val)

        # Rescale adjusted_resale prediction → actual price
        pred_actual = pred * ((months_val // 3 * avg_rpi + BASE_RPI) / 100)

        # Compare against actual resale_price
        y_val_actual = resale.loc[X_val.index, 'resale_price']

        r2s.append(r2_score(y_val_actual, pred_actual))
        maes.append(mean_absolute_error(y_val_actual, pred_actual))
    return np.mean(r2s), np.std(r2s), np.mean(maes), np.std(maes)


# Use adjusted_resale for training
y_act_train_adj = resale.loc[X_act_train.index, 'adjusted_resale']

baseline_params = {'random_state': 42, 'n_jobs': -1}
base_r2, base_std, base_mae, base_mae_std = run_cv_rpi(X_act_train, y_act_train_adj, baseline_params)
print(f"Baseline (default RF) — R²: {base_r2:.4f} ± {base_std:.4f} | MAE: ${base_mae:,.0f} ± ${base_mae_std:,.0f}")

# Sample for search — drop month since run_cv_rpi handles it
X_train_sample, y_train_sample = resample(
    X_act_train, y_act_train_adj,
    n_samples=int(len(X_act_train) * 0.5),
    random_state=42
)
X_train_sample_nm = X_train_sample.drop(columns=['month'])

param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', 0.3, 0.5]
}

search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_dist,
    n_iter=10, cv=3, scoring='r2',
    n_jobs=-1, random_state=42, verbose=1
)
search.fit(X_train_sample_nm, y_train_sample)
print("Best Params:", search.best_params_)

e1_r2, e1_std, e1_mae, e1_mae_std = run_cv_rpi(
    X_act_train, y_act_train_adj,
    {**search.best_params_, 'random_state': 42, 'n_jobs': -1}
)
print(f"RF RandomizedSearch — R²: {e1_r2:.4f} ± {e1_std:.4f} | MAE: ${e1_mae:,.0f} ± ${e1_mae_std:,.0f}")

print("\n--- Tuning Results ---")
print(f"{'Model':<35} {'Mean R²':<12} {'Std R²':<10} {'Mean MAE':<12} {'Std MAE'}")
print(
    f"{'Baseline (default RF)':<35} {base_r2:.4f}{'':>6} {base_std:.4f}{'':>4} ${base_mae:,.0f}{'':>4} ${base_mae_std:,.0f}")
print(f"{'RF RandomizedSearch':<35} {e1_r2:.4f}{'':>6} {e1_std:.4f}{'':>4} ${e1_mae:,.0f}{'':>4} ${e1_mae_std:,.0f}")

Baseline (default RF) — R²: 0.9097 ± 0.0015 | MAE: $43,740 ± $186
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best Params: {'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 0.5, 'max_depth': 30}
RF RandomizedSearch — R²: 0.9123 ± 0.0016 | MAE: $43,204 ± $227

--- Tuning Results ---
Model                               Mean R²      Std R²     Mean MAE     Std MAE
Baseline (default RF)               0.9097       0.0015     $43,740     $186
RF RandomizedSearch                 0.9123       0.0016     $43,204     $227


# Final Model Stability Metrics

In [38]:
best_params = {**search.best_params_, 'random_state': 42, 'n_jobs': -1}

train_r2, train_std, train_mae, train_rmse = run_cv_rpi(
    X_act_train, y_act_train_adj, best_params
)

rf_final = RandomForestRegressor(**best_params)

X_train_final = X_act_train.drop(columns=['month'])
rf_final.fit(X_train_final, y_act_train_adj)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",30
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",5
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",0.5
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples 

In [39]:
y_actual = resale["resale_price"]
X = resale.drop(columns=['floor_area_sqm', 'adjusted_resale', 'resale_price', 'latitude', 'longitude', 'rpi'])

# Convert categorical variables
X = pd.get_dummies(X, drop_first=True)

# Create 5 bins based on price distribution  (low, medium, high for 3 bins)
y_binned = pd.qcut(y, q=5, labels=False)

# First split: Train (60%) and Temp (40%)
X_act_train, X_act_temp, y_act_train, y_act_temp, yb_train, yb_temp = train_test_split(
    X, y_actual, y_binned,
    test_size=0.4,
    stratify=y_binned,
    random_state=42
)

# Second split: Validation (20%) and Test (20%)
X_act_val, X_act_test, y_act_val, y_act_test = train_test_split(
    X_act_temp, y_act_temp,
    test_size=0.5,
    stratify=yb_temp,
    random_state=42
)


In [40]:
def run_cv_rpi(X_tr_full, y_tr_full, model_params, rpi_scaleup=True):
    r2s, maes, rmses = [], [], []
    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    for train_idx, val_idx in kf.split(X_tr_full):
        X_tr = X_tr_full.iloc[train_idx].copy()
        X_val = X_tr_full.iloc[val_idx].copy()
        y_tr = y_tr_full.iloc[train_idx]

        months_val = X_val['month']

        X_tr = X_tr.drop(columns=['month'])
        X_val = X_val.drop(columns=['month'])

        m = RandomForestRegressor(**model_params)
        m.fit(X_tr, y_tr)
        pred = m.predict(X_val)

        if rpi_scaleup:
            pred_actual = pred * ((months_val // 3 * avg_rpi + BASE_RPI) / 100)
        else:
            pred_actual = pred
        y_val_actual = resale.loc[X_val.index, 'resale_price']

        r2s.append(r2_score(y_val_actual, pred_actual))
        maes.append(mean_absolute_error(y_val_actual, pred_actual))
        rmses.append(np.sqrt(mean_squared_error(y_val_actual, pred_actual)))

    return (
        np.mean(r2s), np.std(r2s),
        np.mean(maes), np.std(maes),
        np.mean(rmses), np.std(rmses)
    )

def evaluate_set(X, y_adj, name):
    months = X['month'].copy()
    X_model = X.drop(columns=['month'])

    pred = rf_final.predict(X_model)

    # Rescale
    pred_actual = pred * ((months // 3 * avg_rpi + BASE_RPI) / 100)
    y_actual = resale.loc[X.index, 'resale_price']

    r2 = r2_score(y_actual, pred_actual)
    mae = mean_absolute_error(y_actual, pred_actual)
    rmse = np.sqrt(mean_squared_error(y_actual, pred_actual))

    return r2, mae, rmse

train_r2, train_r2_std, train_mae, train_mae_std, train_rmse, train_rmse_std = run_cv_rpi(
    X_act_train, y_act_train_adj, best_params
)

val_r2, val_mae, val_rmse = evaluate_set(X_act_val, None, "Validation")
test_r2, test_mae, test_rmse = evaluate_set(X_act_test, None, "Test")

print("\nRandom Forest (Hypertuned) Performance")

print("\nTraining Set (5-Fold CV) (Cross-validation Mean ± Std deviation):")
print(f"R2: {train_r2:.4f} ± {train_r2_std:.4f}")
print(f"MAE: {train_mae:,.0f} ± {train_mae_std:,.0f}")
print(f"RMSE: {train_rmse:,.0f} ± {train_rmse_std:,.0f}")

print("\nValidation Set:")
print(f"R2: {val_r2:.4f}")
print(f"MAE: {val_mae:,.0f}")
print(f"RMSE: {val_rmse:,.0f}")

print("\nTest Set:")
print(f"R2: {test_r2:.4f}")
print(f"MAE: {test_mae:,.0f}")
print(f"RMSE: {test_rmse:,.0f}")


Random Forest (Hypertuned) Performance

Training Set (5-Fold CV) (Cross-validation Mean ± Std deviation):
R2: 0.9123 ± 0.0016
MAE: 43,204 ± 227
RMSE: 55,448 ± 345

Validation Set:
R2: 0.9128
MAE: 43,136
RMSE: 55,258

Test Set:
R2: 0.9140
MAE: 43,073
RMSE: 55,130


# Ablation Studies


### Champion Model & Params

In [46]:
champion_params = {
    **search.best_params_,
    'random_state': 42,
    'n_jobs': -1
}

# ── Champion Model ────────────────────────────────────────────────
e1_r2, e1_std, e1_mae, e1_mae_std, e1_rmse, e1_rmse_std = run_cv_rpi(
    X_act_train, y_act_train_adj, champion_params
)
print(f"Champion RF — R²: {e1_r2:.4f} ± {e1_std:.4f} | MAE: ${e1_mae:,.0f} ± ${e1_mae_std:,.0f} | RMSE: {e1_rmse:,.0f} ± ${e1_rmse_std:,.0f} ")

Champion RF — R²: 0.9123 ± 0.0016 | MAE: $43,204 ± $227 | RMSE: 55,448 ± $345 


### Ablation 1: Remove Amenity/Distance Features

In [42]:
amenity_cols = [
    'mrt_count', 'school_count', 'mall_count', 'bus_stop_count',
    'hawker_count', 'wet_market_count',
    'nearest_mrt_dist', 'nearest_bus_stop_dist'
]
X_no_amenity = X_act_train.drop(columns=amenity_cols)
e2_r2, e2_std, e2_mae, e2_mae_std, e2_rmse, e2_rmse_std = run_cv_rpi(
    X_no_amenity, y_act_train_adj, champion_params
)
print(f"Ablation (- Amenity Features) — R²: {e2_r2:.4f} ± {e2_std:.4f} | MAE: ${e2_mae:,.0f} ± ${e2_mae_std:,.0f} | RMSE: ${e2_rmse:,.0f} ± ${e2_rmse_std:,.0f}")

Ablation (- Amenity Features) — R²: 0.8608 ± 0.0028 | MAE: $52,641 ± $303 | RMSE: $69,865 ± $431


### Ablation 2: Remove Storey Type Features

In [43]:
storey_cols = ['storey_type_lower', 'storey_type_middle', 'storey_type_upper']
X_no_storey = X_act_train.drop(columns=storey_cols)
e3_r2, e3_std, e3_mae, e3_mae_std, e3_rmse, e3_rmse_std = run_cv_rpi(
    X_no_storey, y_act_train_adj, champion_params
)
print(f"Ablation (- Storey Features) — R²: {e3_r2:.4f} ± {e3_std:.4f} | MAE: ${e3_mae:,.0f} ± ${e3_mae_std:,.0f} | RMSE: ${e3_rmse:,.0f} ± ${e3_rmse_std:,.0f}")

Ablation (- Storey Features) — R²: 0.9024 ± 0.0020 | MAE: $45,108 ± $273 | RMSE: $58,500 ± $372


### Ablation 3: No RPI descaling

In [44]:
y_no_descaling = resale.loc[X_act_train.index, 'resale_price']
e4_r2, e4_std, e4_mae, e4_mae_std, e4_rmse, e4_rmse_std= run_cv_rpi(
    X_act_train, y_no_descaling, champion_params, False
)
print(f"Ablation (No RPI descaling) — R²: {e4_r2:.4f} ± {e4_std:.4f} | MAE: ${e4_mae:,.0f} ± ${e4_mae_std:,.0f} | RMSE: ${e4_rmse:,.0f} ± ${e4_rmse_std:,.0f}")

Ablation (No RPI descaling) — R²: 0.9215 ± 0.0016 | MAE: $37,130 ± $233 | RMSE: $52,471 ± $471


### Ablation 4: RPI Descaled Prediction

In [45]:
e5_r2 = np.mean(rf_r2)
e5_std = np.std(rf_r2)
e5_mae = np.mean(rf_mae)
e5_mae_std = np.std(rf_mae)

### Results Table

In [51]:
print("\n--- Ablation Results ---")
print(f"{'Experiment':<40} {'Mean R²':<12} {'Std R²':<10} {'Mean MAE':<12} {'Std MAE'}")
print(f"{'Champion RF (Tuned)':<40} {e1_r2:.4f}{'':>6} {e1_std:.4f}{'':>4} ${e1_mae:,.0f}{'':>4} ${e1_mae_std:,.0f}")
print(
    f"{'Ablation 1: - Amenity Features':<40} {e2_r2:.4f}{'':>6} {e2_std:.4f}{'':>4} ${e2_mae:,.0f}{'':>4} ${e2_mae_std:,.0f}")
print(
    f"{'Ablation 2: - Storey Features':<40} {e3_r2:.4f}{'':>6} {e3_std:.4f}{'':>4} ${e3_mae:,.0f}{'':>4} ${e3_mae_std:,.0f}")
print(
    f"{'Ablation 3: - No RPI Descaling':<40} {e4_r2:.4f}{'':>6} {e4_std:.4f}{'':>4} ${e4_mae:,.0f}{'':>4} ${e4_mae_std:,.0f}")
print(
    f"{'Ablation 4: - RPI Descaled Prediction':<40} {e5_r2:.4f}{'':>6} {e5_std:.4f}{'':>4} ${e5_mae:,.0f}{'':>4} ${e5_mae_std:,.0f}")


--- Ablation Results ---
Experiment                               Mean R²      Std R²     Mean MAE     Std MAE
Champion RF (Tuned)                      0.9123       0.0016     $43,204     $227
Ablation 1: - Amenity Features           0.8608       0.0028     $52,641     $303
Ablation 2: - Storey Features            0.9024       0.0020     $45,108     $273
Ablation 3: - No RPI Descaling           0.9215       0.0016     $37,130     $233
Ablation 4: - RPI Descaled Prediction    0.9553       0.0010     $15,702     $60


# Failure/Error Analysis

In [47]:
y_actual = resale["resale_price"]
X = resale.drop(columns=['floor_area_sqm', 'adjusted_resale', 'resale_price', 'latitude', 'longitude', 'rpi'])

# Convert categorical variables
X = pd.get_dummies(X, drop_first=True)

# Create 5 bins based on price distribution  (low, medium, high for 3 bins)
y_binned = pd.qcut(y, q=5, labels=False)

# First split: Train (60%) and Temp (40%)
X_act_train, X_act_temp, y_act_train, y_act_temp, yb_train, yb_temp = train_test_split(
    X, y_actual, y_binned,
    test_size=0.4,
    stratify=y_binned,
    random_state=42
)

# Second split: Validation (20%) and Test (20%)
X_act_val, X_act_test, y_act_val, y_act_test = train_test_split(
    X_act_temp, y_act_temp,
    test_size=0.5,
    stratify=yb_temp,
    random_state=42
)

In [48]:
# Step 1: Predict using main champion model (from hypertuning)
X_test_nm = X_act_test.drop(columns=['month'])
month_test = X_act_test['month']
y_pred_adj = search.best_estimator_.predict(X_test_nm)
y_pred = y_pred_adj * ((month_test // 3 * avg_rpi + BASE_RPI) / 100)
print("Predictions generated ✓")

# Step 2: Build error dataframe
errors_df = X_act_test.copy()
errors_df['actual_price']    = y_act_test.values
errors_df['predicted_price'] = y_pred.values
errors_df['error']           = errors_df['predicted_price'] - errors_df['actual_price']
errors_df['abs_error']       = errors_df['error'].abs()
errors_df['pct_error']       = (errors_df['abs_error'] / errors_df['actual_price']) * 100

# Step 3: Pull back readable columns
meta = resale[['flat_type', 'town']].copy()

errors_df = errors_df.join(meta, how='left')

# Step 4: Top 10 worst predictions
worst = errors_df.nlargest(10, 'abs_error')[[
    'town', 'flat_type',
    'remaining_lease', 'nearest_mrt_dist', 'mrt_count',
    'actual_price', 'predicted_price', 'error', 'pct_error'
]]
print("\n=== TOP 10 WORST PREDICTIONS ===")
print(worst.to_string())

# Step 5: Overshot vs undershot
overshot  = (errors_df['error'] > 0).sum()
undershot = (errors_df['error'] < 0).sum()
print(f"\nOvershot (predicted too high): {overshot}")
print(f"Undershot (predicted too low):  {undershot}")

# Step 6: Error by flat type
print("\n=== MEAN ABS ERROR BY FLAT TYPE ===")
print(errors_df.groupby('flat_type')['abs_error'].mean()
      .sort_values(ascending=False)
      .apply(lambda x: f"${x:,.0f}"))

# Step 7: Error by town (top 10 worst)
print("\n=== MEAN ABS ERROR BY TOWN (top 10 worst) ===")
print(errors_df.groupby('town')['abs_error'].mean()
      .nlargest(10)
      .apply(lambda x: f"${x:,.0f}"))

Predictions generated ✓

=== TOP 10 WORST PREDICTIONS ===
                town flat_type  remaining_lease  nearest_mrt_dist  mrt_count  actual_price  predicted_price          error  pct_error
197373    ANG MO KIO    5 ROOM             1024               455          1     1180000.0     6.133785e+05 -566621.482935  48.018770
210736       PUNGGOL    5 ROOM             1042               124          2     1230000.0     7.632836e+05 -466716.392865  37.944422
198672        BISHAN    4 ROOM              764               243          1     1230000.0     7.636686e+05 -466331.431182  37.913124
156556       PUNGGOL    5 ROOM             1069               171          1     1220000.0     7.640829e+05 -455917.129554  37.370257
202231  CENTRAL AREA    5 ROOM             1015               508          0     1580000.0     1.143301e+06 -436699.094121  27.639183
185805    QUEENSTOWN    5 ROOM              615               739          0     1188000.0     7.552627e+05 -432737.306029  36.425699
1585

# Export Champion Model

In [ ]:
import joblib

# Save the model
joblib.dump(rf_model, 'Models\\champion_model.joblib')